# Whole-volume evaluation — making the 3D vs 2D comparison exact

**The problem this fixes.** Training validates on **centre patches**: one 128³
crop from the middle of each volume. Our 2D baseline of **0.76** was measured
over **whole slices**. Those are not equally hard — a centre crop of a BraTS
volume almost always contains tumour, while whole-volume evaluation includes all
the tissue where a false positive costs you. So the training-time number is
probably flattering.

This notebook re-scores `best.pt` on **complete 240×240×155 volumes** using
sliding-window inference, and accumulates Dice **dataset-level** — summing
intersections and denominators across every validation patient before dividing,
exactly as the 2D metrics were computed. After this, the two numbers measure the
same thing and the comparison needs no asterisk.

**Both outcomes are worth having:**

* stays near the training figure → 3D genuinely beats 2D, quote it cleanly
* drops toward 0.76 → the patch evaluation was flattering, and you say so

---

### Before running

Point `CKPT` at your `best.pt`.

* **Kaggle** — open this as a new notebook, then *Add Data → Your Work → Notebook
  Output* and select the training kernel's output. It appears under
  `/kaggle/input/...`. Also add the BraTS dataset as before.
* **Colab** — it is already in Drive at `/content/drive/MyDrive/mri_3d_unet/best.pt`.

In [ ]:
import os, glob, json, random
import numpy as np, torch, nibabel as nib

# --- point these at your files -------------------------------------------
CKPT = ''        # leave '' to auto-detect
ROOT = ''        # leave '' to auto-detect the BraTS folder
# -------------------------------------------------------------------------

if not CKPT:
    for pat in ('/kaggle/input/**/best.pt',
                '/kaggle/working/checkpoints/best.pt',
                '/content/drive/MyDrive/mri_3d_unet/best.pt',
                './checkpoints/best.pt'):
        hit = glob.glob(pat, recursive=True)
        if hit:
            CKPT = hit[0]; break
if not ROOT:
    hit = glob.glob('/kaggle/input/**/MICCAI_BraTS2020_TrainingData', recursive=True) \
          or glob.glob('/content/data/MICCAI_BraTS2020_TrainingData')
    ROOT = hit[0] if hit else ''

assert CKPT and os.path.exists(CKPT), "set CKPT to your best.pt"
assert ROOT and os.path.exists(ROOT), "set ROOT to the BraTS training folder"
print("checkpoint:", CKPT)
print("data      :", ROOT)

# Pick a device we can actually use. Kaggle hands out either a T4 (sm_75) or a
# P100 (sm_60), and its PyTorch build dropped sm_60 — on a P100 every CUDA call
# fails with "not compatible with the current PyTorch installation". We cannot
# choose the card through the API, so detect it and fall back to CPU rather than
# crashing. Slower, but it always produces the number.
dev = 'cpu'
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    if f'sm_{major}{minor}' in supported:
        try:
            _t = torch.zeros(1, device='cuda') + 1     # prove it really works
            dev = 'cuda'
        except Exception as e:
            print("CUDA present but unusable:", type(e).__name__)
    else:
        print(f"GPU is sm_{major}{minor}; this PyTorch supports {supported}.")
        print("Falling back to CPU — expect a few minutes per patient.")
print("device:", dev, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
ck = torch.load(CKPT, map_location=dev)
cfg = ck.get('config', {})
print(f"trained to epoch {ck['epoch']} | best patch Dice {ck.get('best_dice', 0):.4f}")
print("config:", cfg)

## Rebuild the model and the identical split

The split must reproduce exactly, or we would be evaluating on patients the
model trained on — which would inflate the result badly. Same seed, same
`N_CASES`, same shuffle, all read from the checkpoint's own config so they
cannot drift.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

NUM_CLASSES = 4
MODALITIES  = ['t1', 't1ce', 't2', 'flair']
PATCH       = cfg.get('patch', 128)
BASE_FILT   = cfg.get('base_filters', 16)
N_CASES     = cfg.get('n_cases', 126)
SEED        = cfg.get('seed', 42)
VAL_FRAC    = 0.2

def block(i, o):
    return nn.Sequential(
        nn.Conv3d(i, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True),
        nn.Conv3d(o, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True))

class UNet3D(nn.Module):
    def __init__(self, in_ch=4, n_cls=NUM_CLASSES, f=BASE_FILT):
        super().__init__()
        self.e1, self.e2, self.e3 = block(in_ch, f), block(f, f*2), block(f*2, f*4)
        self.bott = block(f*4, f*8)
        self.u3 = nn.ConvTranspose3d(f*8, f*4, 2, 2); self.d3 = block(f*8, f*4)
        self.u2 = nn.ConvTranspose3d(f*4, f*2, 2, 2); self.d2 = block(f*4, f*2)
        self.u1 = nn.ConvTranspose3d(f*2, f,   2, 2); self.d1 = block(f*2, f)
        self.out = nn.Conv3d(f, n_cls, 1); self.pool = nn.MaxPool3d(2)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        b = self.bott(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b),  e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

model = UNet3D().to(dev)
model.load_state_dict(ck['model']); model.eval()

cases = sorted(d for d in os.listdir(ROOT) if d.startswith('BraTS20'))
dirs = [os.path.join(ROOT, c) for c in cases[:N_CASES]]
dirs = [d for d in dirs if os.path.exists(f"{d}/{os.path.basename(d)}_seg.nii")]
random.Random(SEED).shuffle(dirs)                  # same shuffle as training
n_val = max(1, int(len(dirs) * VAL_FRAC))
val_dirs = dirs[:n_val]
print(f"{len(val_dirs)} validation patients — the same ones the model never trained on")

## Sliding-window inference over the full volume

The model was trained on 128³ patches, so a whole 240×240×155 volume cannot go
through in one pass. Instead we tile it with **50 % overlap** (stride 64),
accumulate the logits, and divide by a count map before taking the argmax.

The overlap matters: predictions are least reliable at a patch's edges, where
the network has no surrounding context. Averaging overlapping windows means
every voxel is predicted at least once from somewhere nearer a patch centre.

In [ ]:
@torch.no_grad()
def predict_volume(x, patch=PATCH, stride=None):
    """x: (4, H, W, D) float32 tensor on device -> (H, W, D) int64 prediction."""
    # 50% overlap on GPU; on CPU that is ~18 tiles per volume and far too
    # slow, so tile without overlap and accept slightly weaker edges.
    stride = stride or (patch // 2 if dev == 'cuda' else patch)
    _, H, W, D = x.shape
    logits = torch.zeros((NUM_CLASSES, H, W, D), device=dev)
    counts = torch.zeros((1, H, W, D), device=dev)

    def starts(n):
        if n <= patch:
            return [0]
        s = list(range(0, n - patch + 1, stride))
        if s[-1] != n - patch:
            s.append(n - patch)          # always cover the far edge
        return s

    for i in starts(H):
        for j in starts(W):
            for k in starts(D):
                sl = (slice(None), slice(i, i+patch), slice(j, j+patch), slice(k, k+patch))
                tile = x[sl].unsqueeze(0)
                pad = [0, max(patch - tile.shape[4], 0),
                       0, max(patch - tile.shape[3], 0),
                       0, max(patch - tile.shape[2], 0)]
                if any(pad):
                    tile = F.pad(tile, pad)
                with torch.amp.autocast('cuda', enabled=(dev == 'cuda')):
                    out = model(tile)[0].float()
                out = out[:, :sl[1].stop-sl[1].start,
                             :sl[2].stop-sl[2].start,
                             :sl[3].stop-sl[3].start]
                logits[:, sl[1], sl[2], sl[3]] += out
                counts[:, sl[1], sl[2], sl[3]] += 1

    return (logits / counts.clamp(min=1)).argmax(0)

def norm(v):
    b = v[v > 0]
    if b.size == 0:
        return v.astype(np.float32)
    return np.clip((v - b.mean()) / (b.std() + 1e-8), -5, 5).astype(np.float32)

print(f"sliding window: {PATCH}^3 patches, stride {PATCH//2} (50% overlap)")

## Run it

Two aggregations are reported, because they answer different questions and
people quote them interchangeably without noticing:

* **Dataset-level** — sum intersections and denominators over *all* patients,
  then divide once. This is how the 2D 0.76 was computed, so **this is the
  comparable number**.
* **Per-case mean** — Dice per patient, then averaged. Gives every patient equal
  weight regardless of tumour size, and is usually a little lower.

Expect roughly 10–30 seconds per patient.

In [ ]:
import time
CLASSES = ['background', 'necrotic', 'oedema', 'enhancing']

inter = np.zeros(NUM_CLASSES); denom = np.zeros(NUM_CLASSES)
per_case = []
t0 = time.time()

for n, d in enumerate(val_dirs, 1):
    cid = os.path.basename(d)
    vols = [norm(nib.load(f"{d}/{cid}_{m}.nii").get_fdata()) for m in MODALITIES]
    gt = nib.load(f"{d}/{cid}_seg.nii").get_fdata().astype(np.int64)
    gt[gt == 4] = 3

    x = torch.from_numpy(np.stack(vols)).to(dev)
    pred = predict_volume(x).cpu().numpy()
    del x; torch.cuda.empty_cache()

    case_d = {}
    for c in range(NUM_CLASSES):
        p, t = (pred == c), (gt == c)
        i_, dn = (p & t).sum(), p.sum() + t.sum()
        inter[c] += i_; denom[c] += dn
        if c > 0:
            case_d[CLASSES[c]] = float(2*i_/dn) if dn > 0 else float('nan')
    per_case.append({'case': cid, **case_d,
                     'mean': float(np.nanmean(list(case_d.values())))})
    print(f"[{n}/{len(val_dirs)}] {cid}  mean {per_case[-1]['mean']:.4f}  "
          f"({time.time()-t0:.0f}s elapsed)", flush=True)

dataset_dice = np.where(denom > 0, 2*inter/np.maximum(denom, 1), np.nan)
print("\n" + "="*58)
print("DATASET-LEVEL Dice (comparable with the 2D numbers)")
for c in range(1, NUM_CLASSES):
    print(f"  {CLASSES[c]:<12} {dataset_dice[c]:.4f}")
mean_ds = float(np.nanmean(dataset_dice[1:]))
print(f"  {'MEAN TUMOUR':<12} {mean_ds:.4f}")
print(f"\nper-case mean (every patient weighted equally): "
      f"{np.nanmean([p['mean'] for p in per_case]):.4f}")

## The verdict

2D reference figures, from `results/segmentation_full_metrics.json`:

| | 2D (whole slices) |
|---|---|
| Mean tumour | **0.76** |
| Enhancing | **0.84** |

Now both sides are measured the same way.

In [ ]:
TWO_D = {'mean': 0.76, 'enhancing': 0.84, 'necrotic': 0.67}

print(f"{'':<14}{'2D':>8}{'3D':>10}{'delta':>10}")
print("-"*42)
rows = [('mean tumour', TWO_D['mean'], mean_ds),
        ('necrotic',    TWO_D['necrotic'], dataset_dice[1]),
        ('enhancing',   TWO_D['enhancing'], dataset_dice[3])]
for name, a, b in rows:
    print(f"{name:<14}{a:>8.3f}{b:>10.3f}{b-a:>+10.3f}")

d = mean_ds - TWO_D['mean']
print()
if d > 0.02:
    print(f"3D WINS by {d:+.3f} on a like-for-like evaluation. Quote it.")
elif d < -0.02:
    print(f"2D wins by {-d:.3f}. Report that — a null result is still a result.")
else:
    print(f"No meaningful difference ({d:+.3f}). The patch-based figure was "
          "flattering; say so.")

out = {'dataset_level': {CLASSES[c]: float(dataset_dice[c]) for c in range(1, 4)},
       'mean_tumour_dice': mean_ds,
       'per_case_mean': float(np.nanmean([p['mean'] for p in per_case])),
       'per_case': per_case,
       'n_val_patients': len(val_dirs),
       'checkpoint_epoch': int(ck['epoch']),
       'patch_based_dice_during_training': float(ck.get('best_dice', 0)),
       'evaluation': 'sliding window, 50% overlap, full 240x240x155 volumes',
       'baseline_2d': TWO_D}

dest = os.path.dirname(CKPT) if os.access(os.path.dirname(CKPT), os.W_OK) else '.'
with open(os.path.join(dest, 'whole_volume_eval.json'), 'w') as f:
    json.dump(out, f, indent=2)
print(f"\nwrote {os.path.join(dest, 'whole_volume_eval.json')}")